# 面试问题：Chunked Prefill 怎样降低 decode 干扰，同时避免长 Prompt 饿死？

**回答主线。** 不切分的长 prefill 会占据一次迭代的大量计算，让已在 decode 的请求出现 TPOT 尖峰。Chunked Prefill 把 prompt 拆成 token 块，每轮优先放 decode，再用剩余 token budget 放一个或多个 prefill chunk，使 decode 搭便车并平滑迭代计算量。

调度器不能只写 `min(chunk_size, remaining)`：还要处理 KV 准入、绝对位置、前缀状态、等待公平、取消回滚、TTFT/TPOT 统计和配置搜索。下面用离散事件模拟，不调用推理服务框架。


In [ ]:
import math  # 导入本单元所需的依赖。
from dataclasses import dataclass, field  # 导入本单元所需的依赖。
from collections import deque  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

# 请求状态显式区分 prefill 进度、decode 进度和时间戳。
@dataclass  # 为下方定义附加声明式配置。
class Request156:  # 定义承载本节状态与行为的数据结构。
    request_id: str  # 执行当前语句以推进本节示例。
    prompt_tokens: int  # 执行当前语句以推进本节示例。
    max_new_tokens: int  # 执行当前语句以推进本节示例。
    arrival_ms: float = 0.0  # 计算并保存当前步骤的中间状态。
    prefilled: int = 0  # 计算并保存当前步骤的中间状态。
    generated: int = 0  # 计算并保存当前步骤的中间状态。
    first_token_ms: float | None = None  # 计算并保存当前步骤的中间状态。
    decode_times: list = field(default_factory=list)  # 计算并保存当前步骤的中间状态。
    cancelled: bool = False  # 计算并保存当前步骤的中间状态。

r156 = Request156("r", 100, 5)  # 计算并保存当前步骤的中间状态。
assert r156.prefilled == 0 and r156.generated == 0  # 用受控断言验证关键不变量。
assert r156.first_token_ms is None  # 用受控断言验证关键不变量。
assert not r156.cancelled  # 用受控断言验证关键不变量。


## 1. 每轮 batch 用统一 token budget 表达

Decode 请求每轮各消费一个新 token；prefill chunk 消费一段 prompt token。调度计划必须记录请求、阶段、绝对起止位置，执行器才能正确追加 KV。


In [ ]:
# WorkItem 用半开区间绑定本轮提交的绝对 token 范围。
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class WorkItem156:  # 定义承载本节状态与行为的数据结构。
    request_id: str  # 执行当前语句以推进本节示例。
    phase: str  # 执行当前语句以推进本节示例。
    start: int  # 执行当前语句以推进本节示例。
    stop: int  # 执行当前语句以推进本节示例。

    @property  # 为下方定义附加声明式配置。
    def tokens(self):  # 定义本节可复用的核心函数。
        return self.stop - self.start  # 返回当前分支计算出的结果。

item156 = WorkItem156("r", "prefill", 0, 32)  # 计算并保存当前步骤的中间状态。
assert item156.tokens == 32  # 用受控断言验证关键不变量。
assert item156.start == 0 and item156.stop == 32  # 用受控断言验证关键不变量。
assert item156.phase in {"prefill", "decode"}  # 用受控断言验证关键不变量。


## 2. Decode-maximal 策略先保护活跃请求的 TPOT

每个已完成 prefill 且未结束的请求先获得一个 decode slot，剩余预算再分给最老 prefill。`chunk_size` 是单请求上限，不应越过本轮总预算或剩余 prompt。


In [ ]:
def schedule_round156(requests, token_budget, chunk_size):  # 定义本节可复用的核心函数。
    # 先按到达时间和 ID 稳定排序 decode，再为 prefill 分配剩余预算。
    plan, remaining = [], token_budget  # 计算并保存当前步骤的中间状态。
    decodes = sorted([r for r in requests if not r.cancelled and r.prefilled == r.prompt_tokens and r.generated < r.max_new_tokens], key=lambda r: (r.arrival_ms, r.request_id))  # 计算并保存当前步骤的中间状态。
    for request in decodes:  # 遍历输入元素以累积或检查结果。
        if remaining < 1:  # 按当前条件选择后续控制路径。
            break  # 调整当前循环或占位控制流。
        plan.append(WorkItem156(request.request_id, "decode", request.generated, request.generated + 1)); remaining -= 1  # 计算并保存当前步骤的中间状态。
    prefills = sorted([r for r in requests if not r.cancelled and r.prefilled < r.prompt_tokens], key=lambda r: (r.arrival_ms, r.request_id))  # 计算并保存当前步骤的中间状态。
    for request in prefills:  # 遍历输入元素以累积或检查结果。
        if remaining <= 0:  # 按当前条件选择后续控制路径。
            break  # 调整当前循环或占位控制流。
        take = min(chunk_size, remaining, request.prompt_tokens - request.prefilled)  # 计算并保存当前步骤的中间状态。
        plan.append(WorkItem156(request.request_id, "prefill", request.prefilled, request.prefilled + take)); remaining -= take  # 计算并保存当前步骤的中间状态。
    return plan  # 返回当前分支计算出的结果。

active156 = Request156("active", 4, 3, prefilled=4)  # 计算并保存当前步骤的中间状态。
long156 = Request156("long", 100, 1)  # 计算并保存当前步骤的中间状态。
plan156 = schedule_round156([active156, long156], token_budget=16, chunk_size=8)  # 计算并保存当前步骤的中间状态。
assert plan156[0].phase == "decode"  # 用受控断言验证关键不变量。
assert sum(x.tokens for x in plan156) <= 16  # 用受控断言验证关键不变量。
assert any(x.request_id == "long" and x.tokens == 8 for x in plan156)  # 用受控断言验证关键不变量。


## 3. 非切分 prefill 会制造 head-of-line blocking

用线性教学成本比较：长 prompt 一次执行时，decode 必须等待整段；切块后每轮只等待一个 chunk。真实 kernel 成本非线性，但尾延迟机制相同。


In [ ]:
def iteration_ms156(prefill_tokens, decode_count, launch_ms=0.1, prefill_ms_token=0.01, decode_ms_token=0.04):  # 定义本节可复用的核心函数。
    # 受控模型把两类工作和固定启动开销相加，仅用于比较调度策略。
    return launch_ms + prefill_tokens * prefill_ms_token + decode_count * decode_ms_token  # 返回当前分支计算出的结果。

unchunked_delay156 = iteration_ms156(4096, 16)  # 计算并保存当前步骤的中间状态。
chunked_delay156 = iteration_ms156(256, 16)  # 计算并保存当前步骤的中间状态。
assert unchunked_delay156 > chunked_delay156  # 用受控断言验证关键不变量。
assert unchunked_delay156 / chunked_delay156 > 5  # 用受控断言验证关键不变量。
assert iteration_ms156(0, 1) > 0  # 用受控断言验证关键不变量。


## 4. 离散事件执行器记录 TTFT 与每次 decode 时间

执行 plan 后才能提交 prefill/decode 进度。首个 decode 完成时间定义 TTFT；相邻 decode 完成时间差形成 TPOT。这里把每轮工作视为并行 batch，共享同一个完成时间。


In [ ]:
def execute_plan156(requests, plan, now_ms):  # 定义本节可复用的核心函数。
    # 先计算整批成本，再原子提交每个 item，避免半批状态泄漏。
    prefill_tokens = sum(i.tokens for i in plan if i.phase == "prefill")  # 计算并保存当前步骤的中间状态。
    decode_count = sum(1 for i in plan if i.phase == "decode")  # 计算并保存当前步骤的中间状态。
    finish = now_ms + iteration_ms156(prefill_tokens, decode_count)  # 计算并保存当前步骤的中间状态。
    by_id = {r.request_id: r for r in requests}  # 计算并保存当前步骤的中间状态。
    for item in plan:  # 遍历输入元素以累积或检查结果。
        request = by_id[item.request_id]  # 计算并保存当前步骤的中间状态。
        if item.phase == "prefill":  # 按当前条件选择后续控制路径。
            assert item.start == request.prefilled  # 用受控断言验证关键不变量。
            request.prefilled = item.stop  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            assert item.start == request.generated  # 用受控断言验证关键不变量。
            request.generated = item.stop  # 计算并保存当前步骤的中间状态。
            request.decode_times.append(finish)  # 执行当前语句以推进本节示例。
            if request.first_token_ms is None:  # 按当前条件选择后续控制路径。
                request.first_token_ms = finish  # 计算并保存当前步骤的中间状态。
    return finish  # 返回当前分支计算出的结果。

test_req156 = Request156("t", 8, 2)  # 计算并保存当前步骤的中间状态。
now156 = execute_plan156([test_req156], [WorkItem156("t", "prefill", 0, 8)], 0.0)  # 计算并保存当前步骤的中间状态。
now156 = execute_plan156([test_req156], [WorkItem156("t", "decode", 0, 1)], now156)  # 计算并保存当前步骤的中间状态。
assert test_req156.prefilled == 8 and test_req156.generated == 1  # 用受控断言验证关键不变量。
assert test_req156.first_token_ms == now156  # 用受控断言验证关键不变量。
assert len(test_req156.decode_times) == 1  # 用受控断言验证关键不变量。


## 5. 持续 decode 流量下要为 prefill 保留进度

绝对 decode 优先可能让新请求永远无法完成 prefill。生产策略可设置最小 prefill 配额、最大等待时间或 deadline。下面从总预算中预留 `prefill_reserve`，其余部分仍保护 decode。


In [ ]:
def fair_schedule156(requests, token_budget, chunk_size, prefill_reserve):  # 定义本节可复用的核心函数。
    # 有等待 prefill 时限制 decode 占用，为最老 prompt 留出确定预算。
    waiting = any(not r.cancelled and r.prefilled < r.prompt_tokens for r in requests)  # 计算并保存当前步骤的中间状态。
    decode_budget = token_budget - min(prefill_reserve, token_budget) if waiting else token_budget  # 计算并保存当前步骤的中间状态。
    decode_plan = schedule_round156([r for r in requests if r.prefilled == r.prompt_tokens], decode_budget, chunk_size)  # 计算并保存当前步骤的中间状态。
    used = sum(i.tokens for i in decode_plan)  # 计算并保存当前步骤的中间状态。
    prefill_plan = schedule_round156([r for r in requests if r.prefilled < r.prompt_tokens], token_budget - used, chunk_size)  # 计算并保存当前步骤的中间状态。
    return decode_plan + prefill_plan  # 返回当前分支计算出的结果。

many_decode156 = [Request156(f"d{i}", 1, 2, prefilled=1) for i in range(10)]  # 计算并保存当前步骤的中间状态。
waiting156 = Request156("waiting", 50, 1)  # 计算并保存当前步骤的中间状态。
fair156 = fair_schedule156(many_decode156 + [waiting156], 8, 4, prefill_reserve=4)  # 计算并保存当前步骤的中间状态。
assert sum(i.tokens for i in fair156) <= 8  # 用受控断言验证关键不变量。
assert any(i.request_id == "waiting" and i.phase == "prefill" for i in fair156)  # 用受控断言验证关键不变量。
assert sum(i.phase == "decode" for i in fair156) <= 4  # 用受控断言验证关键不变量。


## 6. KV 准入按最终长度预留，取消时可回收

只按当前 chunk 分配可能在 prompt 快完成时才发现 decode 无空间。Admission 应估算 `prompt + max_new_tokens` 的逻辑块上限，并用租户配额/物理水位决定接受；执行中只提交已完成的块。


In [ ]:
class KVAdmission156:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, capacity_tokens):  # 定义本节可复用的核心函数。
        self.capacity_tokens = capacity_tokens  # 计算并保存当前步骤的中间状态。
        self.reservations = {}  # 计算并保存当前步骤的中间状态。

    def reserve(self, request):  # 定义本节可复用的核心函数。
        # request_id 幂等，同一请求重试不会重复占用容量。
        need = request.prompt_tokens + request.max_new_tokens  # 计算并保存当前步骤的中间状态。
        if request.request_id in self.reservations:  # 按当前条件选择后续控制路径。
            return True  # 返回当前分支计算出的结果。
        if sum(self.reservations.values()) + need > self.capacity_tokens:  # 按当前条件选择后续控制路径。
            return False  # 返回当前分支计算出的结果。
        self.reservations[request.request_id] = need  # 计算并保存当前步骤的中间状态。
        return True  # 返回当前分支计算出的结果。

    def release(self, request_id):  # 定义本节可复用的核心函数。
        return self.reservations.pop(request_id, 0)  # 返回当前分支计算出的结果。

admission156 = KVAdmission156(120)  # 计算并保存当前步骤的中间状态。
assert admission156.reserve(Request156("a", 80, 20))  # 用受控断言验证关键不变量。
assert not admission156.reserve(Request156("b", 30, 10))  # 用受控断言验证关键不变量。
assert admission156.release("a") == 100 and admission156.reserve(Request156("b", 30, 10))  # 用受控断言验证关键不变量。


## 7. 分块不能重置绝对位置或前缀状态

用一个简单因果递推验证：一次处理完整序列与按 chunk 处理并携带状态应完全一致。Transformer 中对应的是 KV、position id、RoPE 相位和 block table 的连续提交。


In [ ]:
def causal_recurrence156(tokens, initial=0.0, start_position=0):  # 定义本节可复用的核心函数。
    # 位置项依赖全局 start_position，故分块调用必须传递当前位置和状态。
    state, outputs = float(initial), []  # 计算并保存当前步骤的中间状态。
    for offset, token in enumerate(tokens):  # 遍历输入元素以累积或检查结果。
        position = start_position + offset  # 计算并保存当前步骤的中间状态。
        state = 0.7 * state + float(token) + 0.01 * position  # 计算并保存当前步骤的中间状态。
        outputs.append(state)  # 执行当前语句以推进本节示例。
    return np.array(outputs), state  # 返回当前分支计算出的结果。

tokens156 = np.arange(1, 21, dtype=np.float64)  # 计算并保存当前步骤的中间状态。
full_seq156, full_state156 = causal_recurrence156(tokens156)  # 计算并保存当前步骤的中间状态。
first156, state1_156 = causal_recurrence156(tokens156[:7])  # 计算并保存当前步骤的中间状态。
second156, state2_156 = causal_recurrence156(tokens156[7:], state1_156, start_position=7)  # 计算并保存当前步骤的中间状态。
assert np.allclose(np.concatenate([first156, second156]), full_seq156)  # 用受控断言验证关键不变量。
assert math.isclose(state2_156, full_state156)  # 用受控断言验证关键不变量。
assert not np.allclose(causal_recurrence156(tokens156[7:], state1_156, 0)[0], second156)  # 用受控断言验证关键不变量。


## 8. 配置搜索同时验收 TTFT、TPOT、吞吐与公平

Chunk 越小，decode 干扰通常越低，但启动/调度开销更高，长 prompt TTFT 也可能变差。下面用受控 workload 扫描 chunk，输出 p95 TPOT 代理与完成轮数；生产必须重放真实长度和到达分布。


In [ ]:
def simulate156(chunk_size):  # 定义本节可复用的核心函数。
    # 每个配置都新建请求，防止状态从上一次实验泄漏。
    requests = [Request156("long", 128, 4), Request156("short", 8, 6)]  # 计算并保存当前步骤的中间状态。
    now, rounds, durations = 0.0, 0, []  # 计算并保存当前步骤的中间状态。
    while any(r.generated < r.max_new_tokens for r in requests):  # 在终止条件满足前持续推进状态。
        plan = fair_schedule156(requests, token_budget=32, chunk_size=chunk_size, prefill_reserve=8)  # 计算并保存当前步骤的中间状态。
        before = now; now = execute_plan156(requests, plan, now)  # 计算并保存当前步骤的中间状态。
        durations.append(now - before); rounds += 1  # 计算并保存当前步骤的中间状态。
        if rounds > 100:  # 按当前条件选择后续控制路径。
            raise RuntimeError("scheduler made no progress")  # 遇到非法合同立即显式失败。
    return {"chunk": chunk_size, "rounds": rounds, "p95_iteration_ms": float(np.quantile(durations, 0.95)), "finished": all(r.generated == r.max_new_tokens for r in requests)}  # 返回当前分支计算出的结果。

sweep156 = [simulate156(c) for c in (4, 8, 16, 32)]  # 计算并保存当前步骤的中间状态。
assert all(x["finished"] for x in sweep156)  # 用受控断言验证关键不变量。
assert all(x["rounds"] < 100 for x in sweep156)  # 用受控断言验证关键不变量。
assert len({x["p95_iteration_ms"] for x in sweep156}) > 1  # 用受控断言验证关键不变量。


## 面试总结

- Chunked Prefill 把长 prompt 拆块，在同一迭代优先放 decode，并用剩余预算推进 prefill。
- 它改善 TPOT 干扰但可能增加启动开销和长 prompt TTFT，所以 chunk size 必须基于 workload 搜索。
- 调度正确性包括绝对位置/KV 连续、原子进度提交、KV 最终长度准入、取消回收和 starvation 防护。
- 验收同时报告 TTFT、TPOT、goodput、吞吐、等待公平与按长度 slice 的结果。

延伸阅读：[SARATHI](https://arxiv.org/abs/2308.16369)、[Sarathi-Serve](https://arxiv.org/abs/2403.02310)、[Orca Continuous Batching](https://www.usenix.org/conference/osdi22/presentation/yu)。
